# LangChain Agents Tutorial

This notebook demonstrates how to create and use agents in LangChain. We'll build a ReAct agent that can search the web and get weather information.

## Step 1: Setting up the OpenAI API Key

First, we need to set up our OpenAI API key to use the language model.


In [1]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

## Step 2: Installing Required Packages

We need to install the necessary LangChain packages and other dependencies for our agent.


In [2]:
!pip install -q langchain==1.0.5 langchain-groq==1.0.0 langchain-community==0.4.1 langchain-core==1.0.4 requests==2.32.5 duckduckgo-search==8.1.1 ddgs==9.9.0 langsmith==0.4.42

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.2/471.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.9/401.9 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB

In [3]:
!pip show langchain langchain-groq langchain-community langchain-core requests duckduckgo-search ddgs langsmith

Name: langchain
Version: 1.0.5
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-groq
Version: 1.0.0
Summary: An integration package connecting Groq and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/groq
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: groq, langchain-core
Required-by: 
---
Name: langchain-community
Version: 0.4.1
Summary: Community contributed LangChain integrations.
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: aiohttp, dataclasses-json, httpx-sse, langchain-classic, langchain-core, langsmith, numpy, pydantic-settings, PyYAML, requests, SQLAlchemy, tenacity
Required-by: 
---
Name: langchain-core
Version: 1.0

## Step 3: Importing Core Libraries

Let's import the essential LangChain components and other libraries we'll need.


In [4]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
import requests

## Step 4: Setting up the Search Tool

We'll use DuckDuckGo search as one of our tools to allow the agent to search for information on the web.


In [5]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

## Step 5: Creating a Custom Weather Tool

Here we define a custom tool that can fetch weather data for any city using the WeatherStack API. This tool will be available to our agent.


In [ ]:
@tool
def get_weather_data(city: str) -> str:
  """
  This tool fetches the current weather data for a given city
  """
  url = f'https://api.weatherstack.com/current?access_key=[Replace with API Key]]&query={city}' # Enter the weatherstack api key before use

  response = requests.get(url)

  return response.json()

## Step 6: Initializing the Language Model

We create an instance of the ChatOpenAI model that will power our agent's reasoning capabilities.


In [7]:
# llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    # Enable tool calling for this model
    model_kwargs={"tool_choice": "auto"},
    api_key=GROQ_API_KEY
)

## Step 7: Importing Agent Components

We import the necessary components to create a ReAct agent and agent executor.


In [8]:
from langchain.agents import create_agent
from langsmith import Client

In [9]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest

## Step 8: Loading the ReAct Prompt

We pull the standard ReAct (Reasoning and Acting) prompt from LangChain Hub. This prompt template guides the agent on how to reason through problems and take actions.


In [10]:
# Step 2: Pull the ReAct prompt from LangSmith Hub

client = Client()
prompt = client.pull_prompt("hwchase17/react") # pulls the standard ReAct agent prompt
prompt_template_string = prompt.template # Extract the template string from the PromptTemplate object

## Step 9: Creating the ReAct Agent

Now we create our ReAct agent by combining the language model, our tools (search and weather), and the ReAct prompt template.


In [11]:
# Step 3: Create the ReAct agent manually with the pulled prompt
agent = create_agent(
    model=llm,
    tools=[search_tool, get_weather_data],
    system_prompt=prompt_template_string
)

## Step 10: Testing the Agent

Let's test our agent with a complex query that requires both searching for information and getting weather data. The agent will need to:
1. Find the capital of Madhya Pradesh
2. Get the current weather for that city


In [12]:
# Step 5: Invoke
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Find the capital of Madhya Pradesh, then find it's current weather condition"}]}
)

print(response)

{'messages': [HumanMessage(content="Find the capital of Madhya Pradesh, then find it's current weather condition", additional_kwargs={}, response_metadata={}, id='1857cdcb-9d1e-467d-8afc-57bc7ef8fa64'), AIMessage(content='', additional_kwargs={'reasoning_content': "We need to answer: capital of Madhya Pradesh, then current weather condition. Use search for capital, then get weather data for that city. Capital is Bhopal. Then get weather for Bhopal using get_weather_data. Let's do that.", 'tool_calls': [{'id': 'fc_69c7e4b0-1ece-4a65-97d6-07b0f2983c2e', 'function': {'arguments': '{"city":"Bhopal"}', 'name': 'get_weather_data'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 82, 'prompt_tokens': 322, 'total_tokens': 404, 'completion_time': 0.182361703, 'prompt_time': 0.014534949, 'queue_time': 0.033664551, 'total_time': 0.196896652, 'completion_tokens_details': {'reasoning_tokens': 52}}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4867cb6

In [13]:
for messages in response['messages']:
  print(messages)

content="Find the capital of Madhya Pradesh, then find it's current weather condition" additional_kwargs={} response_metadata={} id='1857cdcb-9d1e-467d-8afc-57bc7ef8fa64'
content='' additional_kwargs={'reasoning_content': "We need to answer: capital of Madhya Pradesh, then current weather condition. Use search for capital, then get weather data for that city. Capital is Bhopal. Then get weather for Bhopal using get_weather_data. Let's do that.", 'tool_calls': [{'id': 'fc_69c7e4b0-1ece-4a65-97d6-07b0f2983c2e', 'function': {'arguments': '{"city":"Bhopal"}', 'name': 'get_weather_data'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 82, 'prompt_tokens': 322, 'total_tokens': 404, 'completion_time': 0.182361703, 'prompt_time': 0.014534949, 'queue_time': 0.033664551, 'total_time': 0.196896652, 'completion_tokens_details': {'reasoning_tokens': 52}}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4867cb64c5', 'service_tier': 'on_demand', 'finish_r

## Summary

This notebook demonstrated how to:
- Set up LangChain agents with custom tools
- Use the ReAct framework for reasoning and acting
- Combine multiple tools (search and weather API) in a single agent
- Execute complex multi-step queries that require both information retrieval and data processing

The agent successfully found that Bhopal is the capital of Madhya Pradesh and retrieved its current weather conditions by using both the search tool and the weather API tool in sequence.
